# EXAMEN IIB   
## Aubertin Ochoa  
## 15/07/2026

Instalacion de dependencias

[Enlace al Asistente RAG interactivo en la nube](https://examen-auber-recuperacion.streamlit.app/)

In [1]:
!pip install pandas numpy sentence-transformers chromadb google-generativeai streamlit tqdm chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 69.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 95.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 109.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 83.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━

# A. Preparación del corpus

Importaciones y Configuración de Recursos NLTK

In [1]:
import os
import re
import nltk
import pandas as pd
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def ensure_nltk_resources():
    resources = [
        ('tokenizers/punkt', 'punkt'),
        ('corpora/stopwords', 'stopwords'),
    ]
    for resource_path, package in resources:
        try:
            nltk.data.find(resource_path)
        except LookupError:
            nltk.download(package)

ensure_nltk_resources()
print("Recursos de NLTK verificados y listos.")

Recursos de NLTK verificados y listos.


Definición de Funciones de Limpieza

In [2]:
def clean_arxiv_text(text):
    """
    Limpia el texto eliminando caracteres especiales innecesarios,
    saltos de línea y normalizando espacios, preservando la estructura del inglés.
    """
    if not isinstance(text, str):
        return ""
    
    # 1. Remover saltos de línea y tabulaciones molestos
    text = re.sub(r'\s+', ' ', text)
    
    # 2. Conservar letras, números y puntuación básica relevante para oraciones en inglés
    text = re.sub(r'[^a-zA-Z0-9\s.,;:?!\-\(\)]', '', text)
    
    return text.strip()

def tokenize_and_clean_list(text, language='english', remove_stopwords=False):
    """
    Realiza la tokenización y normalización de un texto plano, con opciones 
    de filtrado alfabético y remoción de palabras vacías (stopwords).
    """
    tokens = word_tokenize(text, language=language)
    tokens = [token.lower() for token in tokens if token.isalpha()]
    
    if remove_stopwords:
        stop_words = set(stopwords.words(language))
        tokens = [token for token in tokens if token not in stop_words]
        
    return tokens

Carga del Dataset Original y unión

In [3]:
from google.colab import drive
import pandas as pd
import os

# Montar Google Drive
drive.mount('/content/drive')

# Rutas de los archivos en Drive (ajusta la ruta según donde los hayas guardado)
file1_path = "/content/drive/MyDrive/Recuperacion/arxiv_data.csv"
file2_path = "/content/drive/MyDrive/Recuperacion/arxiv_data_210930-054931.csv"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# file1_path = "data/arxiv_data.csv"
# file2_path = "data/arxiv_data_210930-054931.csv"

# Lista para almacenar los DataFrames
dfs = []

# Cargar el primer archivo si existe
if os.path.exists(file1_path):
    df1 = pd.read_csv(file1_path)
    dfs.append(df1)
    print(f"Archivo 1 cargado correctamente ({file1_path}): {len(df1)} registros.")
else:
    print(f"Advertencia: No se encontró el Archivo 1 en {file1_path}")

# Cargar el segundo archivo si existe
if os.path.exists(file2_path):
    df2 = pd.read_csv(file2_path)
    dfs.append(df2)
    print(f"Archivo 2 cargado correctamente ({file2_path}): {len(df2)} registros.")
else:
    print(f"Advertencia: No se encontró el Archivo 2 en {file2_path}")

# Concatenar ambos DataFrames si se cargaron con éxito
if len(dfs) > 0:
    df_arxiv = pd.concat(dfs, ignore_index=True)
    print("\n--- UNIÓN COMPLETADA ---")
    print(f"Total de registros combinados: {len(df_arxiv)}")
    print("Columnas disponibles:", df_arxiv.columns)
else:
    raise FileNotFoundError("No se pudo cargar ninguno de los archivos CSV especificados. Verifica las rutas.")

Archivo 1 cargado correctamente (/content/drive/MyDrive/Recuperacion/arxiv_data.csv): 51774 registros.
Archivo 2 cargado correctamente (/content/drive/MyDrive/Recuperacion/arxiv_data_210930-054931.csv): 56181 registros.

--- UNIÓN COMPLETADA ---
Total de registros combinados: 107955
Columnas disponibles: Index(['titles', 'summaries', 'terms', 'abstracts'], dtype='object')


Filtrado de Nulos y Creación de df_cleaned

In [5]:
# Eliminamos registros duplicados que puedan surgir de la unión y nulos en columnas esenciales
df_arxiv = df_arxiv.drop_duplicates().reset_index(drop=True)
df_cleaned = df_arxiv.dropna(subset=['titles', 'abstracts']).reset_index(drop=True)

print(f"Total de registros únicos y sin nulos: {len(df_cleaned)}")

Total de registros únicos y sin nulos: 41127


In [6]:
# Aplicar la limpieza avanzada de texto a títulos y abstracts
df_cleaned['titles_clean'] = df_cleaned['titles'].apply(clean_arxiv_text)
df_cleaned['abstract_clean'] = df_cleaned['abstracts'].apply(clean_arxiv_text)

# Construcción del Texto Enriquecido Final para el RAG
df_cleaned['text_to_embed'] = "Title: " + df_cleaned['titles_clean'] + "\nAbstract: " + df_cleaned['abstract_clean']

# Validamos la generación
print("=== MUESTRA DEL TEXTO ENRIQUECIDO ===")
print(df_cleaned['text_to_embed'].iloc[0])

=== MUESTRA DEL TEXTO ENRIQUECIDO ===
Title: Multi-Level Attention Pooling for Graph Neural Networks: Unifying Graph Representations with Multiple Localities
Abstract: Graph neural networks (GNNs) have been widely used to learn vector representation of graph-structured data and achieved better task performance than conventional methods. The foundation of GNNs is the message passing procedure, which propagates the information in a node to its neighbors. Since this procedure proceeds one step per layer, the range of the information propagation among nodes is small in the lower layers, and it expands toward the higher layers. Therefore, a GNN model has to be deep enough to capture global structural information in a graph. On the other hand, it is known that deep GNN models suffer from performance degradation because they lose nodes local information, which would be essential for good model performance, through many message passing steps. In this study, we propose multi-level attention poo

# B. Representación mediante embeddings  
Inicialización del Modelo de Embeddings y Conexión a ChromaDB

In [23]:
!pip install pandas numpy sentence-transformers chromadb nltk scikit-learn

In [24]:
%pip install chromadb

In [7]:
from sentence_transformers import SentenceTransformer

model_name = "all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(model_name)
print(f"Modelo {model_name} cargado correctamente para la representación vectorial.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Modelo all-MiniLM-L6-v2 cargado correctamente para la representación vectorial.


# Requerimiento C: Almacenamiento y Búsqueda Vectorial
*Configuración del motor de base de datos vectorial persistente mediante ChromaDB para la gestión, almacenamiento e indexación de los embeddings generados.*

In [8]:
import chromadb

chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection_name = "arxiv_papers"

try:
    chroma_client.delete_collection(name=collection_name)
except Exception:
    pass

collection = chroma_client.create_collection(name=collection_name)
print(f"Colección vectorial '{collection_name}' inicializada de forma persistente.")

Colección vectorial 'arxiv_papers' inicializada de forma persistente.


In [9]:
# ==============================================================================
# Indexación Completa por Lotes (Batches) en ChromaDB
# ==============================================================================

# 1. Definir el tamaño del lote (batch size) para no saturar la memoria RAM
BATCH_SIZE = 5000
total_records = len(df_cleaned)

print(f"Iniciando la indexación de los {total_records} documentos en ChromaDB...")

# Procesar secuencialmente en lotes
for start_idx in range(0, total_records, BATCH_SIZE):
    end_idx = min(start_idx + BATCH_SIZE, total_records)
    print(f"\n--- Procesando lote: {start_idx} al {end_idx} (Progreso: {end_idx/total_records:.1%}) ---")
    
    # Extraer el fragmento (lote) actual del DataFrame
    df_batch = df_cleaned.iloc[start_idx:end_idx].copy()
    
    # Preparar listas para ChromaDB de este lote
    batch_documents = df_batch['text_to_embed'].tolist()
    batch_ids = [str(i) for i in df_batch.index]
    
    batch_metadata = [
        {
            "title": row['titles_clean'],
            "abstract": row['abstract_clean']
        }
        for _, row in df_batch.iterrows()
    ]
    
    # Generar embeddings para el lote actual
    # Si tienes GPU disponible, SentenceTransformers la detectará automáticamente
    batch_embeddings = embedding_model.encode(
        batch_documents, 
        batch_size=128,  # Ajuste interno del transformer
        show_progress_bar=True
    )
    
    # Insertar/Guardar el lote actual en la base de datos vectorial
    collection.add(
        embeddings=batch_embeddings.tolist(),
        documents=batch_documents,
        metadatas=batch_metadata,
        ids=batch_ids
    )

print("\n" + "="*50)
print(f"¡ÉXITO TOTAL! Se han indexado {collection.count()} documentos en ChromaDB.")
print("="*50)

Iniciando la indexación de los 41127 documentos en ChromaDB...

--- Procesando lote: 0 al 5000 (Progreso: 12.2%) ---


Batches:   0%|          | 0/40 [00:00<?, ?it/s]


--- Procesando lote: 5000 al 10000 (Progreso: 24.3%) ---


Batches:   0%|          | 0/40 [00:00<?, ?it/s]


--- Procesando lote: 10000 al 15000 (Progreso: 36.5%) ---


...


--- Procesando lote: 35000 al 40000 (Progreso: 97.3%) ---


Batches:   0%|          | 0/40 [00:00<?, ?it/s]


--- Procesando lote: 40000 al 41127 (Progreso: 100.0%) ---


Batches:   0%|          | 0/9 [00:00<?, ?it/s]


¡ÉXITO TOTAL! Se han indexado 41127 documentos en ChromaDB.


# Requerimiento D: Recuperación (Búsqueda Semántica Inicial)
*Implementación del componente de recuperación (Retriever). En esta celda se evalúa el comportamiento del espacio vectorial proyectando una consulta de prueba en lenguaje natural, calculando su embedding y extrayendo los $K=3$ documentos semánticamente más cercanos mediante una consulta de vecinos más próximos en ChromaDB.*

Prueba de Recuperación Semántica (Validación del Retriever)

In [10]:
# Definir una consulta de prueba relacionada con el dataset 
query_test = "How is reinforcement learning used in robotics?"

# 1. Generar el embedding de la consulta
query_embedding = embedding_model.encode([query_test]).tolist()

# 2. Buscar los 3 documentos más cercanos en el espacio vectorial (k=3)
results = collection.query(
    query_embeddings=query_embedding,
    n_results=3
)

# 3. Desplegar los resultados recuperados
print(f"=== RESULTADOS DE BÚSQUEDA PARA: '{query_test}' ===\n")
for i in range(len(results['ids'][0])):
    print(f"Resultado #{i+1} (ID: {results['ids'][0:][0][i]}):")
    print(f"Documento Recuperado:\n{results['documents'][0][i]}")
    print("-" * 50)

=== RESULTADOS DE BÚSQUEDA PARA: 'How is reinforcement learning used in robotics?' ===

Resultado #1 (ID: 10790):
Documento Recuperado:
Title: Gaussian Processes for Data-Efficient Learning in Robotics and Control
Abstract: Autonomous learning has been a promising direction in control and robotics for more than a decade since data-driven learning allows to reduce the amount of engineering knowledge, which is otherwise required. However, autonomous reinforcement learning (RL) approaches typically require many interactions with the system to learn controllers, which is a practical limitation in real systems, such as robots, where many interactions can be impractical and time consuming. To address this problem, current learning approaches typically require task-specific knowledge in form of expert demonstrations, realistic simulators, pre-shaped policies, or specific knowledge about the underlying dynamics. In this article, we follow a different approach and speed up learning by extractin

# Requerimiento E: Generación Aumentada por Recuperación (RAG) - Componente Re-ranker
*Inicialización del modelo Cross-Encoder `ms-marco-MiniLM-L-6-v2`. Este componente actúa en la segunda etapa de la recuperación para evaluar la relación semántica exacta entre la consulta del usuario y los candidatos pre-seleccionados por ChromaDB, mitigando el ruido y ordenando las evidencias de mayor a menor relevancia antes de enviarlas al modelo de generación (LLM).*

In [11]:
from sentence_transformers import CrossEncoder

# Inicializar un modelo Cross-Encoder optimizado para Re-ranking en inglés
reranker_model_name = "ms-marco-MiniLM-L-6-v2"
reranker = CrossEncoder(reranker_model_name)
print(f"Modelo de Re-ranking '{reranker_model_name}' cargado con éxito.")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Modelo de Re-ranking 'ms-marco-MiniLM-L-6-v2' cargado con éxito.


Función de Recuperación de Dos Etapas (Búsqueda Semántica + Re-ranking)

In [12]:
def retrieve_and_rerank(query, collection, embedding_model, reranker, top_n=10, top_k=3):
    """
    1. Recupera top_n documentos usando embeddings densos desde ChromaDB.
    2. Aplica Re-ranking con el Cross-Encoder para quedarse con los mejores top_k.
    """
    # 1. Recuperación inicial en ChromaDB
    query_embedding = embedding_model.encode([query]).tolist()
    initial_results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_n
    )
    
    fetched_docs = initial_results['documents'][0]
    fetched_metadatas = initial_results['metadatas'][0]
    
    if not fetched_docs:
        return [], []

    # 2. Preparar pares para el Cross-Encoder: [(query, doc1), (query, doc2), ...]
    pairs = [[query, doc] for doc in fetched_docs]
    
    # Calcular scores de relevancia con el re-ranker
    rerank_scores = reranker.predict(pairs)
    
    # Combinar documentos, metadatos y sus nuevos puntajes
    scored_docs = list(zip(fetched_docs, fetched_metadatas, rerank_scores))
    
    # Ordenar de mayor a menor puntaje según el Cross-Encoder
    scored_docs.sort(key=lambda x: x[2], reverse=True)
    
    # Seleccionar únicamente los top_k mejores
    final_docs = scored_docs[:top_k]
    
    return final_docs

Integración con el LLM y Generación del Prompt RAG

In [29]:
import os
from google.colab import drive
from google import genai

# ==============================================================================
# 1. MONTAR DRIVE Y CONFIGURAR API KEY DE FORMA SEGURA
# ==============================================================================

# Montar Google Drive si no está montado previamente
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Intentar cargar la API key de manera segura
env_path = "/content/drive/MyDrive/Recuperacion/.env"

if os.path.exists(env_path):
    # Usamos un lector básico de .env si no tienes instalado python-dotenv
    with open(env_path, 'r') as f:
        for line in f:
            if line.strip() and not line.startswith('#'):
                key, value = line.strip().split('=', 1)
                # Limpiar posibles comillas del valor
                os.environ[key.strip()] = value.strip().replace('"', '').replace("'", "")
    print(" Archivo .env cargado exitosamente desde Drive.")
else:
    print(f" Alerta: No se encontró el archivo .env en la ruta: {env_path}")
    print("Asegúrate de haber creado la carpeta 'Recuperacion' en tu Drive y subido el archivo '.env'.")

# Obtener la API key de las variables de entorno ya cargadas
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

 Archivo .env cargado exitosamente desde Drive.


In [16]:
# ==============================================================================
# 2. FUNCIÓN GENERADORA RAG IMPLEMENTADA (LLAMADA REAL AL LLM)
# ==============================================================================

def generate_rag_response(query, context_documents):
    """
    Construye el prompt estructurado inyectando las evidencias recuperadas 
    y realiza la llamada real al modelo de lenguaje (Gemini).
    """
    if not GEMINI_API_KEY:
        raise ValueError("Error: GEMINI_API_KEY no está configurada. Verifica tu archivo .env en Drive.")
        
    # Concatenar los contextos seleccionados por el Re-ranker
    context_text = ""
    for i, (doc, meta, score) in enumerate(context_documents):
        context_text += f"--- Document Evidence #{i+1} (Re-rank Score: {score:.4f}) ---\n"
        context_text += f"Title: {meta.get('title')}\nAbstract: {meta.get('abstract')}\n\n"
        
    # Crear un Prompt robusto protegiendo el sistema contra alucinaciones (Criterio del Examen)
    prompt = f"""
You are an expert scientific research assistant. Answer the user's query using strictly the provided document evidences from arXiv papers. 
If the evidence does not contain enough information to answer the query, clearly state that the corpus does not contain sufficient information.

Document Evidences:
{context_text}

User Query: {query}

Answer:
"""
    
    try:
        # Inicializar el cliente oficial pasándole explícitamente la API Key cargada
        client = genai.Client(api_key=GEMINI_API_KEY)
        
        # Llamar al modelo recomendado (gemini-2.5-flash)
        response = client.models.generate_content(
            model='gemini-2.5-flash', 
            contents=prompt
        )
        return response.text
        
    except Exception as e:
        return f" Error al conectar con la API de Gemini: {e}"

Prueba de Ejecución del Pipeline RAG Completo
Python

In [17]:
# Definir una consulta de prueba formal
query_examen = "How is reinforcement learning used in robotics?"

# 1. Ejecutar Recuperación y Re-ranking
top_reranked_docs = retrieve_and_rerank(
    query=query_examen, 
    collection=collection, 
    embedding_model=embedding_model, 
    reranker=reranker, 
    top_n=10, 
    top_k=3
)

# 2. Armar y mostrar cómo queda el prompt final para el LLM
prompt_listo = generate_rag_response(query_examen, top_reranked_docs)

print("=== PROMPT COMPLETO GENERADO CON EVIDENCIAS RE-ORDENADAS ===")
print(prompt_listo)

=== PROMPT COMPLETO GENERADO CON EVIDENCIAS RE-ORDENADAS ===
Reinforcement learning (RL) is a promising direction for autonomous learning in robotics (Document 2). It is used for learning controllers, though traditional RL approaches can be limited in real robotic systems due to the many interactions typically required, which are impractical and time-consuming (Document 2).

Specific applications and approaches mentioned include:
*   **Addressing limited prior knowledge:** In domains like swarm robotics, where it's difficult for an expert to design a reward function or demonstrate target behavior, preference-based RL frameworks are used. These frameworks exploit expert preferences to learn an approximate policy return, enabling direct policy search (Document 1).
*   **Data-efficient learning:** Model-based policy search methods, such as those that learn a probabilistic Gaussian process transition model, are applied to autonomous learning in real robot tasks to speed up learning by extr

Definición del Set de Consultas de Evaluación (Benchmark de Prueba)

In [23]:
# Definición de consultas para evaluar el comportamiento del sistema
eval_queries = [
    {
        "type": "In-domain (GNN)",
        "query": "What are the main applications of Graph Neural Networks?"
    },
    {
        "type": "In-domain (RL)",
        "query": "How is reinforcement learning used in robotics?"
    },
    {
        "type": "Out-of-domain (Ficticio)",
        "query": "Explain the culinary recipes for cooking traditional Ecuadorian Hornado."
    }
]

print(f"Set de evaluación preparado con {len(eval_queries)} consultas de prueba.")

Set de evaluación preparado con 3 consultas de prueba.


# Requerimiento F: Presentación de Evidencias y Evaluación de la Generación
*La función `run_evaluation_pipeline` sirve para ejecutar y contrastar simultáneamente nuestro sistema contra consultas dentro del dominio (In-domain) y consultas fuera del dominio de arXiv (Out-of-domain) para verificar que el sistema presente las evidencias recuperadas con sus respectivos metadatos y Scores de Re-ranking.*

Función de Ejecución del Pipeline y Reporte de Evidencias

In [32]:
import time
import gc  # Garbage Collector para limpiar la memoria de Colab de inmediato

def run_evaluation_pipeline(queries_list, collection, embedding_model, reranker):
    """
    Ejecuta el flujo completo de RAG limpiando de forma explícita la memoria RAM 
    en cada ciclo para evitar la mezcla de contextos de consultas anteriores.
    """
    total_queries = len(queries_list)
    
    for idx, q_item in enumerate(queries_list):
        q_type = q_item["type"]
        query = q_item["query"]
        
        # 1. Forzar limpieza de variables previas por seguridad
        top_docs = None
        response_or_prompt = None
        gc.collect() 
        
        print("=" * 80)
        print(f"TIPO DE CONSULTA: {q_type}")
        print(f"CONSULTA: '{query}'")
        print("=" * 80)
        
        # 2. Recuperación y Re-ranking (Evidencias específicas de ESTA consulta)
        top_docs = retrieve_and_rerank(
            query=query, 
            collection=collection, 
            embedding_model=embedding_model, 
            reranker=reranker, 
            top_n=8, 
            top_k=3
        )
        
        # 3. Generar el Prompt / Respuesta del LLM
        response_or_prompt = generate_rag_response(query, top_docs)
        
        # 4. Presentación de Resultados y Evidencias (Requerimiento F del Examen)
        print("\n[EVIDENCIAS RECUPERADAS (TOP K RE-RANKED)]")
        for i, (doc, meta, score) in enumerate(top_docs):
            print(f"\nDocumento #{i+1} | Re-rank Score: {score:.4f}")
            print(f"Título: {meta.get('title', 'N/A')}")
            print(f"Abstract Corto: {meta.get('abstract', 'N/A')[:150]}...")
            
        print("\n" + "-"*40)
        print("[RESPUESTA OBTENIDA DEL SISTEMA RAG]")
        print("-"*40)
        print(response_or_prompt)
        print("\n" + "=" * 80 + "\n")
        
        # Pausa de seguridad para evitar el error de cuota (429)
        if idx < total_queries - 1:
            print("⏳ Esperando 15 segundos para la siguiente consulta y vaciando memoria...")
            time.sleep(15)

In [34]:
# ==============================================================================
# EVALUACIÓN INDIVIDUAL: CONSULTA 1
# ==============================================================================
query_1 = "What are the main applications of Graph Neural Networks?"

print(f"Ejecutando Consulta 1: '{query_1}'\n")
docs_1 = retrieve_and_rerank(query_1, collection, embedding_model, reranker, top_n=8, top_k=3)
respuesta_1 = generate_rag_response(query_1, docs_1)

print("[EVIDENCIAS RECUPERADAS]")
for idx, (doc, meta, score) in enumerate(docs_1):
    print(f"↳ Doc #{idx+1} | Score: {score:.4f} | Título: {meta.get('title')}")

print("\n[RESPUESTA DEL SISTEMA]")
print("-" * 50)
print(respuesta_1)

Ejecutando Consulta 1: 'What are the main applications of Graph Neural Networks?'

[EVIDENCIAS RECUPERADAS]
↳ Doc #1 | Score: 5.3526 | Título: Graph Neural Networks for Small Graph and Giant Network Representation Learning: An Overview
↳ Doc #2 | Score: 4.4653 | Título: Graph Neural Networks: Architectures, Stability and Transferability
↳ Doc #3 | Score: 4.3931 | Título: A Practical Guide to Graph Neural Networks

[RESPUESTA DEL SISTEMA]
--------------------------------------------------
Based on the provided document evidences, Graph Neural Networks (GNNs) have applications in:

*   Node and graph classification tasks (Document 1).
*   Recommendation systems (Document 2).
*   Decentralized collaborative control (Document 2).
*   Wireless communication networks (Document 2).


In [35]:
# ==============================================================================
# EVALUACIÓN INDIVIDUAL: CONSULTA 2
# ==============================================================================
query_2 = "How is reinforcement learning used in robotics?"

print(f"Ejecutando Consulta 2: '{query_2}'\n")
docs_2 = retrieve_and_rerank(query_2, collection, embedding_model, reranker, top_n=8, top_k=3)
respuesta_2 = generate_rag_response(query_2, docs_2)

print("[EVIDENCIAS RECUPERADAS]")
for idx, (doc, meta, score) in enumerate(docs_2):
    print(f"↳ Doc #{idx+1} | Score: {score:.4f} | Título: {meta.get('title')}")

print("\n[RESPUESTA DEL SISTEMA]")
print("-" * 50)
print(respuesta_2)

Ejecutando Consulta 2: 'How is reinforcement learning used in robotics?'

[EVIDENCIAS RECUPERADAS]
↳ Doc #1 | Score: 5.0510 | Título: Gaussian Processes for Data-Efficient Learning in Robotics and Control
↳ Doc #2 | Score: 4.8825 | Título: A framework for reinforcement learning with autocorrelated actions
↳ Doc #3 | Score: 4.7543 | Título: Learning Transition Models with Time-delayed Causal Relations

[RESPUESTA DEL SISTEMA]
--------------------------------------------------
Reinforcement learning (RL) is used in robotics for autonomous learning and learning controllers in real robot and control tasks.

Specifically, RL is applied in robotics in several ways:
*   **Data-Efficient Learning and Control:** RL approaches are used to learn controllers for robots, with a focus on improving data-efficiency. This includes learning probabilistic, non-parametric Gaussian process transition models of the system. By explicitly incorporating model uncertainty into long-term planning and controller 

In [36]:
# ==============================================================================
# EVALUACIÓN INDIVIDUAL: CONSULTA 3
# ==============================================================================
query_3 = "Recent advances in diffusion models for image generation."

print(f"Ejecutando Consulta 3: '{query_3}'\n")
docs_3 = retrieve_and_rerank(query_3, collection, embedding_model, reranker, top_n=8, top_k=3)
respuesta_3 = generate_rag_response(query_3, docs_3)

print("[EVIDENCIAS RECUPERADAS]")
for idx, (doc, meta, score) in enumerate(docs_3):
    print(f"↳ Doc #{idx+1} | Score: {score:.4f} | Título: {meta.get('title')}")

print("\n[RESPUESTA DEL SISTEMA]")
print("-" * 50)
print(respuesta_3)

Ejecutando Consulta 3: 'Recent advances in diffusion models for image generation.'

[EVIDENCIAS RECUPERADAS]
↳ Doc #1 | Score: 7.5805 | Título: Score Matching Model for Unbounded Data Score
↳ Doc #2 | Score: 3.2747 | Título: Toward Spatially Unbiased Generative Models
↳ Doc #3 | Score: 2.8496 | Título: Trainable Nonlinear Reaction Diffusion: A Flexible Framework for Fast and Effective Image Restoration

[RESPUESTA DEL SISTEMA]
--------------------------------------------------
Recent advances in diffusion models for image generation include:

*   **Incorporation of Stochastic Differential Equations (SDEs)**: This approach has led to state-of-the-art performance in image generation tasks. (Document Evidence #1)
*   **Analysis and Improvements at Zero Diffusion Time**: Advances have been made by analyzing diffusion models at the zero diffusion time (t=0), where the score function can diverge and score estimation might fail. (Document Evidence #1)
*   **Unbounded Diffusion Model (UDM)**: 

In [37]:
# ==============================================================================
# EVALUACIÓN INDIVIDUAL: CONSULTA 4
# ==============================================================================
query_4 = "Techniques for improving retrieval-augmented generation systems."

print(f"Ejecutando Consulta 4: '{query_4}'\n")
docs_4 = retrieve_and_rerank(query_4, collection, embedding_model, reranker, top_n=8, top_k=3)
respuesta_4 = generate_rag_response(query_4, docs_4)

print("[EVIDENCIAS RECUPERADAS]")
for idx, (doc, meta, score) in enumerate(docs_4):
    print(f"↳ Doc #{idx+1} | Score: {score:.4f} | Título: {meta.get('title')}")

print("\n[RESPUESTA DEL SISTEMA]")
print("-" * 50)
print(respuesta_4)

Ejecutando Consulta 4: 'Techniques for improving retrieval-augmented generation systems.'

[EVIDENCIAS RECUPERADAS]
↳ Doc #1 | Score: 1.1976 | Título: Hybrid Retrieval-Generation Reinforced Agent for Medical Image Report Generation
↳ Doc #2 | Score: -3.9435 | Título: A Retrieve-and-Edit Framework for Predicting Structured Outputs
↳ Doc #3 | Score: -5.1470 | Título: Image Retrieval with Mixed Initiative and Multimodal Feedback

[RESPUESTA DEL SISTEMA]
--------------------------------------------------
Based on the provided document evidences, techniques for improving retrieval-augmented generation systems include:

*   **Hybrid Retrieval-Generation Framework:** Reconciling traditional retrieval-based approaches with modern learning-based approaches (Document Evidence #1).
*   **Hierarchical Decision-Making:** Employing a procedure where a high-level retrieval policy module chooses to either retrieve a template sentence from a database or invoke a low-level generation module to generate 

### **Análisis de los Resultados (Consulta por Consulta)**

* **Consulta 1: Graph Neural Networks (GNN)**
* **Estado:** **Excelente (Muy Bien).**
El espacio vectorial recuperó documentos sumamente relevantes (todos con scores altos $>4.3$). El LLM resumió de forma sintética las aplicaciones reales y colocó las citas bibliográficas (`Document 1`, `Document 2`) en perfecta concordancia con los abstracts.


* **Consulta 2: Reinforcement Learning en Robótica**
* **Estado:** **Excelente (Muy Bien).**
La respuesta es técnicamente brillante. El LLM no solo entendió la teoría, sino que extrajo problemas físicos reales de la robótica (como evitar el "shaking" o temblor de los motores usando acciones autocorrelacionadas de la evidencia #2), demostrando una fidelidad absoluta al texto recuperado.


* **Consulta 3: Diffusion Models para Imágenes**
* **Estado:** **Excelente (Muy Bien).**
El primer documento obtuvo un score altísimo de **`7.5805`**, lo que demuestra la excelente precisión del Re-ranker. El LLM tradujo conceptos matemáticos complejos de SDEs y límites en tiempo cero ($t=0$) de forma coherente y sin alucinar.


* **Consulta 4: Mejora de Sistemas RAG**
* **Estado:** **Correcto con Limitación de Corpus (Bien).**
Es la consulta más reveladora. Al ser un dataset antiguo que no tiene papers de RAG moderno, el Re-ranker asignó scores muy bajos y negativos (`-3.94` y `-5.14`) advirtiendo la falta de relación directa. Aun así, el sistema funcionó bien porque el LLM **no inventó técnicas modernas (LangChain, LlamaIndex, etc.)**, sino que se limitó estrictamente a explicar lo recuperado (*Retrieve-and-Edit* de código y plantillas médicas).